# Validation comparison across recommendation models

This notebook compares the seven model variants produced by notebooks 01–05.
Every result must refer to the canonical **chronological split within each user**:
the earliest 60% of that user's ratings train the model, the next 20% support
validation, and the latest 20% remain reserved for testing. Rounding applies to
the cumulative 60% and 80% boundaries. Equal timestamps use movie ID as a
deterministic tie breaker.

The comparison uses macro mean Recall@10: compute recall separately for each
user with at least one validation rating of 4.0 or higher, then weight users
equally. All positive held-out movies remain in the denominator, including
movies absent from the training candidate catalog. Candidates come only from
training interactions, and every previously rated training movie is excluded.

Run notebooks 01–05 before this notebook, then run notebook 07. Final testing
occurs at the end of notebook 07, after hybrid tuning and model selection.
This order keeps test outcomes out of validation decisions.


## Load the shared evaluation contract

The loader checks raw-data hashes and reads the exact split files written by
notebook 01. A separate artifact directory prevents old global-date results
from entering this experiment. The test partition is intentionally not loaded.


In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10

# Resolve the project from either its root or any nested notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the project containing data/raw/ml-latest-small.")

raw_dir = data_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to write the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} does not match the split manifest.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]

train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]

def save_validation_results(results, filename):
    """Attach the split and evaluation contract to each per-user result row."""
    saved = results.copy()
    saved["split_id"] = SPLIT_ID
    saved["split_fingerprint"] = split_manifest["split_fingerprint"]
    saved["evaluation_split"] = "validation"
    saved["positive_threshold"] = POSITIVE_THRESHOLD
    saved["k"] = TOP_K
    saved.to_csv(processed_dir / filename, index=False)

def validate_result_provenance(results, label="results"):
    expected = {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }
    for column, value in expected.items():
        assert column in results and results[column].eq(value).all(), (
            f"{label}: missing or incompatible {column}; rerun its source notebook."
        )

print("Split:", SPLIT_ID)
print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))

import numpy as np


Split: per_user_chronological_60_20_20_v1
Training ratings: 60255
Validation ratings: 20164


## Load and validate each model's per-user results

Matching row counts alone would miss stale experiments or different recall
denominators. Check the split fingerprint, positive threshold, list length,
user identities, positive counts and arithmetic against the canonical
validation data before averaging any scores. Users without positive training
history remain in the evaluation and receive each model's documented fallback.


In [2]:
result_files = {
    "Popularity": "popularity_validation_results.csv",
    "Item-based CF": "item_based_validation_results.csv",
    "Genres": "genre_validation_results.csv",
    "Original TF-IDF": "tfidf_validation_results.csv",
    "Log-TF TF-IDF": "log_tfidf_validation_results.csv",
    "Embeddings": "embedding_validation_results.csv",
    "Adjusted embeddings": "adjusted_embedding_validation_results.csv",
}
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
]
expected_counts = (
    validation_positives.groupby("userId")["movieId"].nunique().sort_index()
)
if expected_counts.empty:
    raise ValueError("There are no positive validation ratings to evaluate.")

def checked_validation_results(filename, label):
    results = pd.read_csv(processed_dir / filename)
    validate_result_provenance(results, label)
    if label in {"Original TF-IDF", "Log-TF TF-IDF", "Embeddings", "Adjusted embeddings"}:
        catalog_path = processed_dir / "training_plot_catalog.csv"
        catalog_manifest = json.loads(
            (processed_dir / "plot_catalog_manifest.json").read_text(encoding="utf-8")
        )
        assert catalog_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
        expected_catalog_hash = catalog_manifest["files"][catalog_path.name]["sha256"]
        assert file_sha256(catalog_path) == expected_catalog_hash
        assert "plot_catalog_sha256" in results and results["plot_catalog_sha256"].eq(
            expected_catalog_hash
        ).all(), f"{label}: plot catalog changed; rerun its source notebook"
    assert results["userId"].is_unique, f"{label}: repeated evaluated users"
    results = results.set_index("userId").sort_index()
    assert results.index.equals(expected_counts.index), f"{label}: evaluated users differ"
    assert results["positive_movie_count"].eq(expected_counts).all(), (
        f"{label}: recall denominators differ from canonical validation positives"
    )
    assert results["hits_at_10"].between(0, TOP_K).all(), label
    assert results["hits_at_10"].le(results["positive_movie_count"]).all(), label
    assert results["hits_at_10"].mod(1).eq(0).all(), label
    assert np.allclose(
        results["recall_at_10"],
        results["hits_at_10"] / results["positive_movie_count"],
    ), f"{label}: saved recall is inconsistent with hits and positives"
    expected_history = results.index.isin(train_ratings["userId"])
    assert "has_training_history" in results, label
    assert results["has_training_history"].eq(expected_history).all(), label
    return results

evaluations = {
    label: checked_validation_results(filename, label)
    for label, filename in result_files.items()
}


## Compare validation performance

The table reports equal-weight mean recall and the number of recovered
positive movies. It deliberately makes no fixed claim about a winning model;
all values are recomputed from the revised split. The saved table is an input
for review, while notebook 07 performs the final selection after adding hybrid
weight candidates.


In [3]:
comparison_rows = []
for label, results in evaluations.items():
    comparison_rows.append({
        "model": label,
        "users": len(results),
        "mean_recall_at_10": results["recall_at_10"].mean(),
        "total_hits_at_10": int(results["hits_at_10"].sum()),
        "users_with_training_history": int(results["has_training_history"].sum()),
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["mean_recall_at_10", "model"], ascending=[False, True])
    .reset_index(drop=True)
)
display(model_comparison.assign(
    mean_recall_at_10=model_comparison["mean_recall_at_10"].map("{:.4%}".format)
))
save_validation_results(model_comparison, "validation_model_comparison.csv")


,model,users,mean_recall_at_10,total_hits_at_10,users_with_training_history
0,Item-based CF,594,11.2240%,590,594
1,Popularity,594,7.2092%,391,594
2,Original TF-IDF,594,1.1116%,65,594
3,Adjusted embeddings,594,0.9738%,68,594
4,Genres,594,0.8460%,67,594
5,Log-TF TF-IDF,594,0.8393%,39,594
6,Embeddings,594,0.5384%,22,594


## Quantify the candidate-set limitation

Recall can be low even when every recommendable positive movie is retrieved:
a movie absent from training cannot be recommended by these models. Report
the share of positive validation movies present in the shared training
catalog without removing unavailable movies from the denominator. This is a
catalog-availability diagnostic, not a replacement recall metric; the top-ten
list imposes an additional retrieval limit.


In [4]:
candidate_ids = set(train_ratings["movieId"])
coverage_rows = []
for user_id, positives in validation_positives.groupby("userId"):
    available = int(positives["movieId"].isin(candidate_ids).sum())
    positive_count = len(positives)
    coverage_rows.append({
        "userId": user_id,
        "positive_movie_count": positive_count,
        "available_positive_count": available,
        "available_positive_fraction": available / positive_count,
        "maximum_possible_recall_at_10": min(TOP_K, available) / positive_count,
    })
validation_candidate_coverage = pd.DataFrame(coverage_rows)
display(validation_candidate_coverage.head())
print("Users evaluated:", len(expected_counts))
print("Users without positive validation ratings:",
      validation_ratings["userId"].nunique() - len(expected_counts))
print("Mean fraction of positives in training catalog:",
      validation_candidate_coverage["available_positive_fraction"].mean())
print("Mean achievable Recall@10 given catalog and list size:",
      validation_candidate_coverage["maximum_possible_recall_at_10"].mean())
save_validation_results(validation_candidate_coverage, "validation_candidate_coverage.csv")


,userId,positive_movie_count,available_positive_count,available_positive_fraction,maximum_possible_recall_at_10
0,1,40,40,1.000000,0.250000
1,2,5,5,1.000000,1.000000
2,3,5,3,0.600000,0.600000
3,4,19,17,0.894737,0.526316
4,5,5,5,1.000000,1.000000


Users evaluated: 594
Users without positive validation ratings: 16
Mean fraction of positives in training catalog: 0.9668661943437494
Mean achievable Recall@10 given catalog and list size: 0.775358345535806


## Interpretation boundary

The revised split evaluates recommendations for users with existing histories.
It is not a new-user benchmark. A user may have training ratings without any
ratings above the positive threshold, or without any liked movies that have
usable plots; those are distinct fallback cases.

Chronology is enforced within each user's history. Another user's training
rating can occur after the target user's validation or test rating. Together
with current genre/plot snapshots and a pretrained language encoder, this
makes the project an offline preference-retrieval benchmark rather than a
simulation of one historical production deployment. Differences in validation
scores guide model selection; they are not independent estimates of final
generalization after tuning.
